# HDEM Sub-Ensemble Performance Evaluation — HCMUT Dataset

This notebook measures the performance of each individual sub-ensemble in HDEM,
compared against the full HDEM ensemble.

**Sub-Ensemble Configuration:**
- Sub-Ensemble 1: ExtraTrees, RandomForest, XGBoost
- Sub-Ensemble 2: RandomForest, MLP, GradientBoosting
- Sub-Ensemble 3: Lasso, XGBoost, ExtraTrees
- Meta-Learner: GradientBoosting

In [1]:
import os
import sys
sys.path.append(os.path.abspath(".."))
import pandas as pd
import numpy as np
import time
from preprocessing import *
from HDEM_sub_eval import SubEnsembleEvaluator

# set_seed(42)

In [2]:
df = pd.read_csv(r'../output_csv/HCMUT-SuperNodeXP-2017-1.0.swf.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15886 entries, 0 to 15885
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   job_id                    15886 non-null  float64
 1   submit_time               15886 non-null  float64
 2   wait_time                 15886 non-null  float64
 3   run_time                  15886 non-null  float64
 4   num_allocated_processors  15886 non-null  float64
 5   avg_cpu_time_used         15886 non-null  float64
 6   used_memory               15886 non-null  float64
 7   requested_processors      15886 non-null  float64
 8   requested_time            15886 non-null  float64
 9   requested_memory          15886 non-null  float64
 10  status                    15886 non-null  float64
 11  user_id                   15886 non-null  float64
 12  group_id                  15886 non-null  float64
 13  executable_id             15886 non-null  float64
 14  queue_

In [3]:
feature_columns = ['requested_processors', 'requested_time', 'avg_cpu_time_used', 'used_memory', 'submit_time', 'wait_time', 'user_id', 'group_id', 'executable_id', 'queue_id']
target_column = 'run_time'

X_train, X_val, X_test, Y_train, Y_val, Y_test, scaler = prepare_data_DL(df, feature_columns, target_column)

In [4]:
# Collect results across iterations
hdem_metrics = {'RMSE': [], 'MAE': [], 'R2': [], 'Inference Time': []}
sub_metrics = {
    'sub_ensemble1': {'RMSE': [], 'MAE': [], 'R2': [], 'Inference Time': []},
    'sub_ensemble2': {'RMSE': [], 'MAE': [], 'R2': [], 'Inference Time': []},
    'sub_ensemble3': {'RMSE': [], 'MAE': [], 'R2': [], 'Inference Time': []}
}

NUM_ITER = 5

for iter in range(NUM_ITER):
    print(f"\n{'='*60}")
    print(f"Iteration {iter + 1} / {NUM_ITER}")
    print(f"{'='*60}")

    evaluator = SubEnsembleEvaluator(X_train, X_val, X_test, Y_train, Y_val, Y_test, scaler)
    evaluator.meta_model_name = 'gradientboosting'
    model_combinations = [
        ['extratrees', 'randomforest', 'xgboost'],
        ['randomforest', 'mlp', 'gradientboosting'],
        ['lasso', 'xgboost', 'extratrees']
    ]
    evaluator.num_sub = len(model_combinations)
    evaluator.init_base_sub(model_combinations)

    hdem_result, sub_results = evaluator.run_model_with_sub_eval()

    # Collect HDEM metrics
    hdem_metrics['RMSE'].append(hdem_result['RMSE'])
    hdem_metrics['MAE'].append(hdem_result['MAE'])
    hdem_metrics['R2'].append(hdem_result['R² Score'])
    hdem_metrics['Inference Time'].append(hdem_result['Inference Time'])

    # Collect sub-ensemble metrics
    for sub_key in sub_metrics:
        sub_metrics[sub_key]['RMSE'].append(sub_results[sub_key]['RMSE'])
        sub_metrics[sub_key]['MAE'].append(sub_results[sub_key]['MAE'])
        sub_metrics[sub_key]['R2'].append(sub_results[sub_key]['R² Score'])
        sub_metrics[sub_key]['Inference Time'].append(sub_results[sub_key]['Inference Time'])


Iteration 1 / 5
Key on Test Set
Predict on Test Set
MAE:  17685.30997818213
RMSE:  88762.14289272192
R2:  0.7712615153343195
Inference Time:  2.496464307918105e-05

Sub-Ensemble Individual Performance

sub_ensemble1 (extratrees, randomforest, xgboost):
  MAE:  20159.0859
  RMSE: 97038.5686
  R²:   0.7266
  Inference Time: 0.000008s/sample

sub_ensemble2 (randomforest, mlp, gradientboosting):
  MAE:  16716.8783
  RMSE: 78036.1562
  R²:   0.8232
  Inference Time: 0.000008s/sample

sub_ensemble3 (lasso, xgboost, extratrees):
  MAE:  41673.3458
  RMSE: 119128.4762
  R²:   0.5880
  Inference Time: 0.000006s/sample

Comparison Summary
Model                                                 RMSE          MAE         R²
-------------------------------------------------------------------------------
sub_ensemble1 (extratrees, randomforest, xgboost)   97038.5686   20159.0859     0.7266
sub_ensemble2 (randomforest, mlp, gradientboosting)   78036.1562   16716.8783     0.8232
sub_ensemble3 (lasso, x

In [ ]:
# Save results to CSV
for sub_key in sub_metrics:
    results_df = pd.DataFrame(sub_metrics[sub_key])
    results_df.to_csv(f'../output_HCMUT/{sub_key}_results.csv', index=False)

hdem_df = pd.DataFrame(hdem_metrics)
hdem_df.to_csv('../output_HCMUT/HDEM_with_sub_eval_results.csv', index=False)

# Print final summary
print(f"\n{'='*80}")
print(f"FINAL SUMMARY — HCMUT Dataset ({NUM_ITER} iterations)")
print(f"{'='*80}")
print(f"{'Model':<50} {'RMSE':>10} {'MAE':>10} {'R²':>10}")
print("-" * 80)

sub_names = {
    'sub_ensemble1': 'Sub-Ens 1 (ExtraTrees, RF, XGB)',
    'sub_ensemble2': 'Sub-Ens 2 (RF, MLP, GB)',
    'sub_ensemble3': 'Sub-Ens 3 (Lasso, XGB, ExtraTrees)'
}

for sub_key, label in sub_names.items():
    m = sub_metrics[sub_key]
    print(f"{label:<50} {np.mean(m['RMSE']):>10.2f} {np.mean(m['MAE']):>10.2f} {np.mean(m['R2']):>10.4f}")

print("-" * 80)
print(f"{'HDEM (Full Dynamic Ensemble)':<50} {np.mean(hdem_metrics['RMSE']):>10.2f} {np.mean(hdem_metrics['MAE']):>10.2f} {np.mean(hdem_metrics['R2']):>10.4f}")


FINAL SUMMARY — HCMUT Dataset (5 iterations)
Model                                                    RMSE        MAE         R²
--------------------------------------------------------------------------------
Sub-Ens 1 (ExtraTrees, RF, XGB)                      97046.00   20167.30     0.7266
Sub-Ens 2 (RF, MLP, GB)                              79216.16   17216.52     0.8176
Sub-Ens 3 (Lasso, XGB, ExtraTrees)                  119128.48   41673.35     0.5880
--------------------------------------------------------------------------------
HDEM (Full Dynamic Ensemble)                         89026.89   17935.08     0.7699


: 